In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from statsmodels.tsa.stattools import adfuller, kpss

In [ ]:
# Show all rows
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_rows', 1000)

- DRCC =  Credit Card Delinquency rate (all commercial banks)
- COR = Credit card charge-off rate
- TERM = Terms on Credit Card plans

In [ ]:
corc_df = pd.read_csv('../data/CORCCACBS.csv')
drcc_df = pd.read_csv('../data/DRCCLACBS.csv')
term_df = pd.read_csv('../data/TERMCBCCALLNS.csv')

In [ ]:
print(corc_df.head(15))
print(drcc_df[drcc_df['date']>'1994'])
print(term_df.head(15))

Organize our data - it currently has multiple entries for each date + has a hours/minutes/second. I'll make a function to make this more streamlined

In [ ]:

def fix_df(df):

    a=df.groupby(df['date'])['value'].mean().round(2)
    
    b = pd.to_datetime(df['date'].unique())

    df = pd.DataFrame({
        'date':b,
        'value':a.values
    })

    df = df.set_index('date')

    return df



In [ ]:
drcc_df = fix_df(drcc_df)
corc_df = fix_df(corc_df)
term_df = fix_df(term_df)
term_df = term_df.dropna()

In [ ]:
print(drcc_df.head(1))
print(corc_df.head(1))
print(term_df.head(1))
print(len(drcc_df),len(corc_df),len(term_df))

Plotting them out 
- They all start at different dates, so I'll start them all at 1995 to have an equal starting date for easier comparison

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)

drcc_df.loc['1995':]['value'].plot(ax=axes[0], title='Delinquency Rate (DRCCLACBS)', color='steelblue')
corc_df.loc['1995':]['value'].plot(ax=axes[1], title='Charge-off Rate (CORCCACBS)', color='coral')
term_df.loc['1995':]['value'].plot(ax=axes[2], title='Credit Card Terms (TERMCBCCALLNS)', color='green')

plt.tight_layout()
plt.show()

Finding correlational value between variables vs. outcome

In [ ]:
print(len(drcc_df.loc['1995':]['value']),drcc_df.loc['1995':]['value'].tail(4))
print(len(corc_df.loc['1995':]['value']),corc_df.loc['1995':]['value'].tail(4))
print(len(term_df.loc['1995':'2025']['value']),term_df.loc['1995':]['value'].tail(4))


term_corr = term_df.loc['1995':]['value'].ffill().asfreq('QS').last()
drcc_corr = drcc_df.loc['1995':]['value'].asfreq('QS').last()



print(term_corr.isna().sum())



print('\n','The Correlation between Charge off rate and Delinquency rate is:',drcc_df.loc['1995':]['value'].corr(corc_df.loc['1995':]['value']))
print('\n','Correlation between delinquency and term length is:',term_corr.corr(drcc_corr))



#Figure out a way to match

We have a time mismatch clearly, how should I split?
A couple of ideas:

- Align our data to yearly data, that way we'll have more aligned data points but probably not the best idea will be mising out on (4!!) predictions per year, since the data we're looking for, the delinquency rate, is calculated quarterly.

- try to match where we can? maybe align closes data to that data point ??

- let Pandas try to match the timelines as best it can

Let pandas merge

In [ ]:
df_multi = pd.concat([drcc_df,corc_df,term_df], axis =1 , join='outer')

In [ ]:
# we chose 1995, since thats the earliest date that they all have data

df_multi = df_multi.loc["1995":]
print(len(df_multi),df_multi.head(10))


df_multinfiltered = df_multi.asfreq('QS').dropna()

print(len(df_multinfiltered))


#this had a much higher count since term had data starting at a much earlier date than the other two.

without term this time:
also Changing the name of Column from value to distinct name 

In [ ]:
drcc_df.rename(columns={'value': 'delinquency'}, inplace=True)
corc_df.rename(columns={'value': 'charge_off'}, inplace=True)

In [ ]:
df_multi1 = pd.concat([drcc_df,corc_df], axis =1 , join='outer')
df_multi1=df_multi1.asfreq('QS').dropna()
print(len(df_multi1),df_multi1.head(10))

## TERMCBCCALLNS — dropped

- Excluded due to time cycle mismatch -> quarterly cycle (QS-NOV vs QS-JAN) 
- data loss on merge (249 to 0 observations) after dropping Null



#### Next, we'll split into train val test

also introduce a feature engineered variable, 'anamoly flag'

In [50]:
def traintestsplit(df):
    train = df.loc[:'2015']
    val = df.loc['2016':'2019']
    test = df.loc['2020':]

    return train,val ,test

def featureng(df):
    df_fe = df.copy()
    df_fe['anamoly_flag'] = ((df_fe['charge_off'] > 5) | 
                          (df_fe['charge_off'] < 2.0)).astype(int)
    return df_fe

df_multi1_fe = featureng(df_multi1)
# this only includes delinquency rates as well as charge off rates. Now we'll split
train_mv, val_mv, test_mv = traintestsplit(df_multi1_fe)





In [51]:
print(len(train_mv[train_mv['anamoly_flag']==1])) # 35 points in time where we had 'anamoly' rates!

35


p,d,q

P,D,Q,s


p,q,P,Q usually fairly small typically 0-2 but sometimes up to 5 or 6

d,D typically 0/1

s - is our main seasonl pattern's period

In [ ]:
def objective_cat(trial):
    params = {
        "iterations": 1000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.5, 1.0),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 50),
        "loss_function": "Logloss",
        "eval_metric": "AUC",          
        "random_seed": 11,              
        "verbose": 0,
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=11)
    auc_scores = []

    
    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]

        model = CatBoostClassifier(**params)
        
        
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            early_stopping_rounds=50,
            verbose=0
        )

        
        preds = model.predict_proba(X_val)[:, 1]
        fold_auc = roc_auc_score(y_val, preds)
        auc_scores.append(fold_auc)

    return np.mean(auc_scores)


study_cat = optuna.create_study(direction="maximize")
optuna.logging.set_verbosity(optuna.logging.WARNING)
study_cat.optimize(objective_cat, n_trials=50, timeout=600)

print("Best AUC:", study_cat.best_value)
print("Best params:")
for k, v in study_cat.best_params.items():
    print(f"  {k}: {v}")